In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import joblib
import os


In [2]:
df = pd.read_csv("../data/application_train.csv")

xgb_model = joblib.load("../outputs/xgboost_model.pkl")

In [3]:
y = df["TARGET"]

X = df.drop(columns=["TARGET", "SK_ID_CURR"])

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_test.shape

(61503, 120)

In [5]:
default_probability = xgb_model.predict_proba(X_test)[:, 1]

default_probability[:10]

array([0.38257182, 0.30814868, 0.793484  , 0.41731572, 0.47624585,
       0.67519605, 0.16183233, 0.11064539, 0.8348802 , 0.5074259 ],
      dtype=float32)

In [6]:
def probability_to_score(probability, min_score=300, max_score=850):
    """
    Converts default probability into a credit score.

    Higher default probability -> lower score.
    Lower default probability -> higher score.
    """
    score = max_score - probability * (max_score - min_score)
    return np.round(score).astype(int)

In [7]:
credit_scores = probability_to_score(default_probability)

credit_scores[:10]

array([640, 681, 414, 620, 588, 479, 761, 789, 391, 571])

In [8]:
def assign_risk_tier(score):
    """
    Assigns a risk tier based on credit score.
    """
    if score >= 750:
        return "Very Low Risk"
    elif score >= 700:
        return "Low Risk"
    elif score >= 650:
        return "Medium Risk"
    elif score >= 600:
        return "High Risk"
    else:
        return "Very High Risk"

In [9]:
results = X_test.copy()

results["actual_default"] = y_test.values
results["default_probability"] = default_probability
results["credit_score"] = credit_scores
results["risk_tier"] = results["credit_score"].apply(assign_risk_tier)

results[["actual_default", "default_probability", "credit_score", "risk_tier"]].head(20)

,actual_default,default_probability,credit_score,risk_tier
256571,0,0.382572,640,High Risk
191493,0,0.308149,681,Medium Risk
103497,0,0.793484,414,Very High Risk
130646,0,0.417316,620,High Risk
211898,0,0.476246,588,Very High Risk
130549,0,0.675196,479,Very High Risk
287975,0,0.161832,761,Very Low Risk
144906,0,0.110645,789,Very Low Risk
102173,0,0.834880,391,Very High Risk
254495,1,0.507426,571,Very High Risk


In [10]:
results["credit_score"].describe()

count    61503.000000
mean       623.573094
std        109.393193
min        330.000000
25%        543.000000
50%        636.000000
75%        713.000000
max        842.000000
Name: credit_score, dtype: float64

In [11]:
results["risk_tier"].value_counts()

risk_tier
Very High Risk    24257
Low Risk          10373
Medium Risk       10145
High Risk          8870
Very Low Risk      7858
Name: count, dtype: int64

In [12]:
results["risk_tier"].value_counts(normalize=True)

risk_tier
Very High Risk    0.394404
Low Risk          0.168658
Medium Risk       0.164951
High Risk         0.144221
Very Low Risk     0.127766
Name: proportion, dtype: float64

In [14]:
tier_order = [
    "Very Low Risk",
    "Low Risk",
    "Medium Risk",
    "High Risk",
    "Very High Risk"
]

tier_analysis["risk_tier"] = pd.Categorical(
    tier_analysis["risk_tier"],
    categories=tier_order,
    ordered=True
)

tier_analysis = tier_analysis.sort_values("risk_tier")

tier_analysis

,risk_tier,applicants,defaults,default_rate,avg_default_probability,avg_credit_score
4,Very Low Risk,7858,113,0.014380,0.133357,776.654110
1,Low Risk,10373,256,0.024679,0.228846,724.135448
2,Medium Risk,10145,372,0.036668,0.318681,674.730310
0,High Risk,8870,531,0.059865,0.409530,624.759414
3,Very High Risk,24257,3693,0.152245,0.619725,509.150183


In [15]:
def credit_decision(score):
    """
    Basic credit decision based on score.
    """
    if score >= 700:
        return "Approve"
    elif score >= 600:
        return "Manual Review"
    else:
        return "Reject"

In [16]:
results["credit_decision"] = results["credit_score"].apply(credit_decision)

results[[
    "actual_default",
    "default_probability",
    "credit_score",
    "risk_tier",
    "credit_decision"
]].head(20)

,actual_default,default_probability,credit_score,risk_tier,credit_decision
256571,0,0.382572,640,High Risk,Manual Review
191493,0,0.308149,681,Medium Risk,Manual Review
103497,0,0.793484,414,Very High Risk,Reject
130646,0,0.417316,620,High Risk,Manual Review
211898,0,0.476246,588,Very High Risk,Reject
130549,0,0.675196,479,Very High Risk,Reject
287975,0,0.161832,761,Very Low Risk,Approve
144906,0,0.110645,789,Very Low Risk,Approve
102173,0,0.834880,391,Very High Risk,Reject
254495,1,0.507426,571,Very High Risk,Reject


In [17]:
decision_analysis = results.groupby("credit_decision").agg(
    applicants=("actual_default", "count"),
    defaults=("actual_default", "sum"),
    default_rate=("actual_default", "mean"),
    avg_credit_score=("credit_score", "mean"),
    avg_default_probability=("default_probability", "mean")
).reset_index()

decision_analysis

,credit_decision,applicants,defaults,default_rate,avg_credit_score,avg_default_probability
0,Approve,18231,369,0.020240,746.772256,0.187688
1,Manual Review,19015,903,0.047489,651.420195,0.361059
2,Reject,24257,3693,0.152245,509.150183,0.619725


In [18]:
os.makedirs("../outputs", exist_ok=True)

results[[
    "actual_default",
    "default_probability",
    "credit_score",
    "risk_tier",
    "credit_decision"
]].to_csv("../outputs/scored_applicants_sample.csv", index=False)

In [19]:
os.path.exists("../outputs/scored_applicants_sample.csv")

True

## Credit score engine results

This notebook converts XGBoost default probabilities into an interpretable credit score ranging from 300 to 850.

The score is designed so that higher predicted default probabilities result in lower credit scores, while lower predicted default probabilities result in higher credit scores. Applicants are then assigned to risk tiers ranging from Very Low Risk to Very High Risk.

A basic credit decision layer was also added. Applicants with scores above 700 are approved, applicants between 600 and 699 are sent to manual review, and applicants below 600 are rejected.

This step transforms the project from a pure machine learning model into an initial credit decision engine. The next step is to improve the score calibration, add business rules, and explain individual decisions using SHAP.